# Dataset overview

Purpose: establish dataset coverage for Chapter 4 by reporting the observed hardware, model, workload, and experiment-category composition without assuming a fixed number of rows.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

## Dataset shape

The shape is reported directly from the input so that rerunning the notebook after a legitimate dataset update does not require changing the analysis.

In [ ]:
shape = pd.DataFrame({"measure": ["rows", "columns"], "value": [data.shape[0], data.shape[1]]})
display(shape)
save_table(shape, "01_dataset_shape.csv")

## Hardware coverage

This table shows the number of executions and successful executions observed for each hardware platform.

In [ ]:
hardware_coverage = (data.assign(hardware_label=data["hardware"].map({h: h for h in data["hardware"].unique()}))
    .groupby("hardware", as_index=False)
    .agg(executions=("experiment_id", "size"), successful=("status", lambda s: s.eq("success").sum()), models=("model", "nunique"), workloads=("workload", "nunique")))
hardware_coverage["success_rate"] = hardware_coverage["successful"] / hardware_coverage["executions"]
display(hardware_coverage)
save_table(hardware_coverage, "01_hardware_coverage.csv")

## Model coverage

Model coverage is reported across hardware and workload combinations to make gaps in the experimental matrix visible.

In [ ]:
model_coverage = (data.groupby("model", as_index=False)
    .agg(executions=("experiment_id", "size"), hardware=("hardware", "nunique"), workloads=("workload", "nunique"), successful=("status", lambda s: s.eq("success").sum()), model_size_b=("model_size_b", "first"), architecture=("architecture", "first"), quantization=("quantization", "first")))
display(model_coverage)
save_table(model_coverage, "01_model_coverage.csv")

## Workload coverage

The workload table documents the number of observations available for each task category.

In [ ]:
workload_coverage = (data.groupby("workload", as_index=False)
    .agg(executions=("experiment_id", "size"), hardware=("hardware", "nunique"), models=("model", "nunique"), successful=("status", lambda s: s.eq("success").sum())))
display(workload_coverage)
save_table(workload_coverage, "01_workload_coverage.csv")

## Experiment category distribution

Experiment categories are inherited from the reproducible dataset-generation step and are summarized here to distinguish CPU, full-offload, partial-offload, and MoE-offload observations.

In [ ]:
category_coverage = data.groupby("experiment_category", as_index=False).agg(executions=("experiment_id", "size"), successful=("status", lambda s: s.eq("success").sum()))
display(category_coverage)
save_table(category_coverage, "01_experiment_category_coverage.csv")